In [21]:
%load_ext autoreload
%autoreload 2
%load_ext rpy2.ipython

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


In [22]:
import pandas as pd

import src
from src.load import DataLoader

r_colormap = src.r_colormap
r_out = str(src.OUT)
pd.options.display.float_format = "{:.1f}".format

In [23]:
%%R -i r_colormap -i r_out

suppressMessages(library(tidyverse))
library(ggplot2)
library(ggeffects)
library(here)
library(ggpubr)

options(scipen = 999)

cmap <- setNames(r_colormap$color, r_colormap$channel)

# Load Data

In [24]:
dl = DataLoader()

videos = (
    dl.channels()
    .join(dl.videos(filtered=True, _ignore_sentence_filter=True), "channel_id")
    .to_pandas()
)
sents = dl.sentences(filtered=True).join(dl.popbert(filtered=True), "sentence_id").to_pandas()

sents = sents.groupby("video_id", observed=True).agg(
    n_sents=("video_id", "size"),
    n_elite=("elite", "sum"),
    n_pplcentr=("pplcentr", "sum"),
    avg_elite=("elite", "mean"),
    avg_pplcentr=("pplcentr", "mean"),
)

# Dataset Summary Table

In [25]:
channel_overview = (
    videos.loc[videos.video_was_live == False]
    .merge(sents, on="video_id", how="left")
    .groupby("channel", observed=True)
    .agg(
        chFollowers=("channel_follower_count", "first"),
        videos=("video_was_live", lambda x: (x == 0).sum()),
        avgVideoLen=("video_duration", "mean"),
        avgViews=("video_views", "mean"),
        avgLikes=("video_likes", "mean"),
        disabledLikes=("video_likes", lambda x: x.isna().sum()),
        nSentences=("n_sents", "sum"),
        # avg_comments=("video_comment_count", lambda x: x.dropna().mean()),
        first_video=("video_datetime_upload", "min"),
        latest_video=("video_datetime_upload", "max"),
    )
)

In [26]:
channel_overview

,chFollowers,videos,avgVideoLen,avgViews,avgLikes,disabledLikes,nSentences,first_video,latest_video
channel,,,,,,,,,
AfD BT,515000,6208,409.0,53650.7,4392.7,0,345690.0,2017-12-06 13:23:54,2025-02-04 18:00:08
AfD TV,320000,1956,520.0,57202.5,4499.6,113,160178.0,2017-12-08 00:21:22,2025-02-03 15:19:09
CDU,28400,955,498.1,19814.0,147.0,2,67916.0,2017-12-11 16:28:36,2025-02-04 22:08:58
CSU,6340,178,420.0,20806.7,40.8,4,12653.0,2017-12-14 21:19:06,2025-01-12 10:00:06
FDP,26900,847,475.4,28505.2,105.0,846,52806.0,2018-01-06 16:02:32,2025-02-04 17:56:50
Greens,32900,634,601.2,15239.0,162.5,6,49107.0,2018-01-12 10:16:37,2025-02-04 20:52:23
Left,72900,770,548.9,16597.9,937.1,4,56687.0,2017-12-11 14:33:45,2025-02-04 18:06:03
SPD,31600,1106,571.6,9133.1,164.9,2,85532.0,2017-12-07 12:17:56,2025-02-03 11:34:41


In [27]:
# sum of durations

sum_of_seconds = videos.video_duration.sum()
print(f"Total sum of video durations: {round(sum_of_seconds / 60 / 60, 2)} hours")

Total sum of video durations: 6704.5 hours


In [28]:
# number of videos

count_videos = videos.video_id.size
print(f"Total number of valid videos: {count_videos}")

Total number of valid videos: 14655


In [30]:
# number of sentencs

count_sents = channel_overview.nSentences.sum()
print(f"Total number of valid sentences: {count_sents}")

Total number of valid sentences: 830569.0


In [31]:
summary_table = channel_overview.drop(["first_video", "latest_video"], axis=1).T

summary_table

channel,AfD BT,AfD TV,CDU,CSU,FDP,Greens,Left,SPD
chFollowers,515000.0,320000.0,28400.0,6340.0,26900.0,32900.0,72900.0,31600.0
videos,6208.0,1956.0,955.0,178.0,847.0,634.0,770.0,1106.0
avgVideoLen,409.0,520.0,498.1,420.0,475.4,601.2,548.9,571.6
avgViews,53650.7,57202.5,19814.0,20806.7,28505.2,15239.0,16597.9,9133.1
avgLikes,4392.7,4499.6,147.0,40.8,105.0,162.5,937.1,164.9
disabledLikes,0.0,113.0,2.0,4.0,846.0,6.0,4.0,2.0
nSentences,345690.0,160178.0,67916.0,12653.0,52806.0,49107.0,56687.0,85532.0


In [32]:
path = src.OUT / "tables/dataset_summary.csv"
path.unlink(missing_ok=True)
summary_table.to_csv(path)

# View Count Violin Plot

In [ ]:
df = videos.merge(sents, on="video_id")

In [ ]:
%%R -i df -w 1000 -h 600

df_plot <- df %>%
   mutate(
      likes = video_likes + 1,
      views = video_views + 1,
)

view_plot = ggplot(df_plot, aes(x=channel, y=views, fill=channel)) +
   geom_boxplot(alpha=0.6) +
   geom_violin(alpha=0.3, trim=T, scale="width") +
   scale_y_continuous(trans="log10", breaks=scales::breaks_log(n=5)) +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 21
   ) +
   theme(
      axis.text.x=element_text(angle=20, hjust=1),
      legend.position = "none"
   ) +
   xlab("Channel") +
   ylab("log10(ViewCount)")

like_plot = ggplot(df_plot, aes(x=channel, y=likes, fill=channel)) +
   geom_boxplot(alpha=0.6) +
   geom_violin(alpha=0.3, trim=T, scale="width") +
   scale_y_continuous(trans="log10", breaks=scales::breaks_log(n=8)) +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 21
   ) +
   theme(
      axis.text.x=element_text(angle=20, hjust=1),
      legend.position = "none"
   ) +
   xlab("Channel") +
   ylab("log10(LikeCount)")


ggarrange(view_plot, like_plot, ncol=2)

p <- here(r_out, "/figures/view_count.svg")
if (file.exists(p)) file.remove(p)
ggsave(p)

Saving 13.9 x 8.33 in image


In addition: Warning messages:
1: Removed 760 rows containing non-finite outside the scale range
(`stat_boxplot()`). 
2: Removed 760 rows containing non-finite outside the scale range
(`stat_ydensity()`). 
3: Groups with fewer than two datapoints have been dropped.
ℹ Set `drop = FALSE` to consider such groups for position adjustment purposes. 


# Populism Amount Plot

In [ ]:
%%R -i df -w 1000 -h 600

df_plot <- df %>%
   mutate(
      elite = (n_elite / n_sents * 100) + 1,
      pplcentr = (n_pplcentr / n_sents * 100) + 1,
)

elite_plot = ggplot(df_plot, aes(x=channel, y=elite, fill=channel)) +
   geom_boxplot(alpha=0.6, outliers=F, coef=0.5) +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 22
   ) +
   theme(
      axis.text.x=element_text(angle=20, hjust=1),
      legend.position = "none"
   ) +
   xlab("Channel") +
   ylab("% Anti-Elitism")

pplcentr_plot = ggplot(df_plot, aes(x=channel, y=pplcentr, fill=channel)) +
   geom_boxplot(alpha=0.6, outliers=F, coef=0.5) +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 22
   ) +
   theme(
      axis.text.x=element_text(angle=20, hjust=1),
      legend.position = "none"
   ) +
   xlab("Channel") +
   ylab("% People-Centrism")


ggarrange(elite_plot, pplcentr_plot, ncol=2)

p <- here(r_out, "/figures/populism_per_party.svg")
if (file.exists(p)) file.remove(p)
ggsave(p)

Saving 13.9 x 8.33 in image
